# RT-DETR R101 on AWS Inferentia2 with Custom NKI Grid Sample Kernel

**Model**: `PekingU/rtdetr_r101vd_coco_o365` (76M params, 80 COCO classes, object detection)  
**Hardware**: inf2.8xlarge (1 NeuronDevice, 2 NeuronCores)  
**SDK**: Neuron SDK 2.28+ (torch-neuronx 2.9, NKI Beta 2)  

## Why a Custom Kernel?

RT-DETR uses deformable attention with `F.grid_sample`, which is unsupported on Neuron.
The standard workaround (`torch.gather`) causes DMA thrashing.
A custom NKI kernel using hardware DMA gather (`vector_offset` indirect addressing) delivers
a significant speedup by reading 128 spatial positions per hardware instruction.

## Instance Setup

This notebook runs on an **inf2.8xlarge** instance with the Deep Learning AMI Neuron (Ubuntu 24.04).  
Activate the pre-installed venv:
```bash
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate
pip install jupyter transformers pillow requests
```

## 1. Environment Check

In [1]:
import sys
import os
import time
import subprocess

# NKI requires explicit platform target during trace()
os.environ.setdefault("NEURON_PLATFORM_TARGET_OVERRIDE", "gen2")

import torch
import torch.nn as nn
import torch_neuronx
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"torch_neuronx: {torch_neuronx.__version__}")
print(f"Python: {sys.version}")

result = subprocess.run(["neuron-ls"], capture_output=True, text=True)
print(result.stdout)

PyTorch: 2.9.0+cu128
torch_neuronx: 2.9.0.2.12.22436+0f1dac25
Python: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
instance-type: inf2.8xlarge
instance-id: i-0256febd895e22e28
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 2      | 0-1      | 32 GB  | 0000:00:1f.0 | 0-31     | -1   |
+--------+--------+----------+--------+--------------+----------+------+



## 2. NKI Grid Sample Kernel

This cell contains the complete NKI kernel, host-side helpers, and two monkey-patches
required for compatibility with `torch_neuronx.trace()`.

### Architecture

1. **Host side** (PyTorch): Convert normalized grid coords to pixel coords.
   Compute 4 bilinear corner indices and weights. Pad spatial dim to multiple of 128.

2. **NKI kernel**: For each tile of 128 output spatial points:
   - Load 4 index tiles (128 uint32 values each) into SBUF
   - 4x DMA gather with `vector_offset`: each reads 128 rows of C channels from HBM
   - Load 4 weight tiles into SBUF
   - Weighted sum via `nisa.activation(scale=weight)` + `nisa.tensor_tensor(op=add)`
   - Store result to HBM

### Key NKI Details

- **Must return HBM-allocated output**: Writing to input params does NOT propagate to XLA.
- **`nisa.tensor_tensor` does NOT broadcast on CoreV3+**: Use `nisa.activation(scale=w, bias=zero_bias)` instead.
- **PlaceholderTensor patches**: During `torch_neuronx.trace()`, tensors are PlaceholderTensor
  objects. Two patches make NKI recognize them as torch.Tensor.

In [2]:
import nki
import nki.language as nl
import nki.isa as nisa

# ── Monkey-patch 1: Make NKI recognize PlaceholderTensor as torch.Tensor ──
import nki.compile as _nki_compile

_orig_check_args = _nki_compile._check_args

def _patched_check_args(*args):
    def build_typename(a):
        t = type(a)
        module = getattr(t, "__module__", None)
        qualname = getattr(t, "__qualname__", None)
        if isinstance(a, torch.Tensor) and module and not module.startswith("torch."):
            return "torch.Tensor"
        return f"{module}.{qualname}"
    return {build_typename(arg) for arg in args}

_nki_compile._check_args = _patched_check_args

# ── Monkey-patch 2: Convert PlaceholderTensors for NKI C extension ──
from nki.compiler.backends.neuron import TraceKernel as _TraceKernelModule

_TraceKernelClass = _TraceKernelModule.TraceKernel
_orig_specialize = _TraceKernelClass.specialize_and_call

def _patched_specialize(self, boundargs, output_path_prefix=None):
    def _to_regular_tensor(t):
        if isinstance(t, torch.Tensor) and type(t).__module__.startswith("torch_neuronx"):
            return torch.empty(t.shape, dtype=t.dtype)
        return t
    if boundargs.args:
        new_args = tuple(_to_regular_tensor(a) for a in boundargs.args)
        params = list(boundargs.signature.parameters.keys())
        for i, param_name in enumerate(params):
            if i < len(new_args):
                boundargs.arguments[param_name] = new_args[i]
    return _orig_specialize(self, boundargs, output_path_prefix)

_TraceKernelClass.specialize_and_call = _patched_specialize

print("NKI monkey-patches applied.")


# ── NKI Kernel ──

@nki.jit
def nki_bilinear_grid_sample_kernel(
    input_flat,   # (H*W, C) flattened spatial feature map
    idx_00,       # (num_out_padded, 1) int32 corner indices
    idx_01,
    idx_10,
    idx_11,
    w_00,         # (num_out_padded, 1) float32 bilinear weights
    w_01,
    w_10,
    w_11,
):
    """NKI kernel for bilinear grid sampling using DMA gather."""
    num_out = idx_00.shape[0]
    C = input_flat.shape[1]
    P_MAX = 128
    num_tiles = num_out // P_MAX

    output = nl.ndarray((num_out, C), dtype=input_flat.dtype, buffer=nl.hbm)

    # Hoist constant zero bias outside loop
    zero_bias = nl.ndarray((P_MAX, 1), dtype=nl.float32, buffer=nl.sbuf)
    nisa.memset(dst=zero_bias, value=0.0)

    for tile_idx in nl.affine_range(num_tiles):
        tile_start = tile_idx * P_MAX

        # Load corner indices into SBUF
        idx00_sb = nl.ndarray((P_MAX, 1), dtype=nl.uint32, buffer=nl.sbuf)
        idx01_sb = nl.ndarray((P_MAX, 1), dtype=nl.uint32, buffer=nl.sbuf)
        idx10_sb = nl.ndarray((P_MAX, 1), dtype=nl.uint32, buffer=nl.sbuf)
        idx11_sb = nl.ndarray((P_MAX, 1), dtype=nl.uint32, buffer=nl.sbuf)

        nisa.dma_copy(dst=idx00_sb, src=idx_00[tile_start:tile_start + P_MAX, 0:1])
        nisa.dma_copy(dst=idx01_sb, src=idx_01[tile_start:tile_start + P_MAX, 0:1])
        nisa.dma_copy(dst=idx10_sb, src=idx_10[tile_start:tile_start + P_MAX, 0:1])
        nisa.dma_copy(dst=idx11_sb, src=idx_11[tile_start:tile_start + P_MAX, 0:1])

        # DMA gather: 4 bilinear corners (128 rows x C channels each)
        val00_sb = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)
        val01_sb = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)
        val10_sb = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)
        val11_sb = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)

        nisa.dma_copy(dst=val00_sb, src=input_flat.ap(
            [[C, P_MAX], [1, C]], offset=0, vector_offset=idx00_sb, indirect_dim=0))
        nisa.dma_copy(dst=val01_sb, src=input_flat.ap(
            [[C, P_MAX], [1, C]], offset=0, vector_offset=idx01_sb, indirect_dim=0))
        nisa.dma_copy(dst=val10_sb, src=input_flat.ap(
            [[C, P_MAX], [1, C]], offset=0, vector_offset=idx10_sb, indirect_dim=0))
        nisa.dma_copy(dst=val11_sb, src=input_flat.ap(
            [[C, P_MAX], [1, C]], offset=0, vector_offset=idx11_sb, indirect_dim=0))

        # Load weights
        w00_sb = nl.ndarray((P_MAX, 1), dtype=nl.float32, buffer=nl.sbuf)
        w01_sb = nl.ndarray((P_MAX, 1), dtype=nl.float32, buffer=nl.sbuf)
        w10_sb = nl.ndarray((P_MAX, 1), dtype=nl.float32, buffer=nl.sbuf)
        w11_sb = nl.ndarray((P_MAX, 1), dtype=nl.float32, buffer=nl.sbuf)

        nisa.dma_copy(dst=w00_sb, src=w_00[tile_start:tile_start + P_MAX, 0:1])
        nisa.dma_copy(dst=w01_sb, src=w_01[tile_start:tile_start + P_MAX, 0:1])
        nisa.dma_copy(dst=w10_sb, src=w_10[tile_start:tile_start + P_MAX, 0:1])
        nisa.dma_copy(dst=w11_sb, src=w_11[tile_start:tile_start + P_MAX, 0:1])

        # Bilinear blend: output = w00*v00 + w01*v01 + w10*v10 + w11*v11
        tmp = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)
        prod = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)

        nisa.activation(dst=tmp, op=nl.copy, data=val00_sb, scale=w00_sb, bias=zero_bias)
        nisa.activation(dst=prod, op=nl.copy, data=val01_sb, scale=w01_sb, bias=zero_bias)
        nisa.tensor_tensor(dst=tmp, data1=tmp, data2=prod, op=nl.add)
        nisa.activation(dst=prod, op=nl.copy, data=val10_sb, scale=w10_sb, bias=zero_bias)
        nisa.tensor_tensor(dst=tmp, data1=tmp, data2=prod, op=nl.add)
        nisa.activation(dst=prod, op=nl.copy, data=val11_sb, scale=w11_sb, bias=zero_bias)
        nisa.tensor_tensor(dst=tmp, data1=tmp, data2=prod, op=nl.add)

        nisa.dma_copy(dst=output[tile_start:tile_start + P_MAX, 0:C], src=tmp)

    return output


# ── Host-Side Helpers ──

def _compute_bilinear_indices_weights(input_tensor, grid):
    """Convert normalized grid coords to bilinear interpolation indices and weights."""
    N, C, H, W = input_tensor.shape
    H_out, W_out = grid.shape[1], grid.shape[2]
    num_out = H_out * W_out

    grid_x, grid_y = grid[..., 0], grid[..., 1]
    pix_x = ((grid_x + 1.0) * W - 1.0) / 2.0
    pix_y = ((grid_y + 1.0) * H - 1.0) / 2.0

    x0, y0 = pix_x.floor().long(), pix_y.floor().long()
    x1, y1 = x0 + 1, y0 + 1
    wx1, wy1 = pix_x - x0.float(), pix_y - y0.float()
    wx0, wy0 = 1.0 - wx1, 1.0 - wy1

    valid_x0 = (x0 >= 0) & (x0 < W)
    valid_y0 = (y0 >= 0) & (y0 < H)
    valid_x1 = (x1 >= 0) & (x1 < W)
    valid_y1 = (y1 >= 0) & (y1 < H)

    w00 = (wx0 * wy0) * (valid_x0 & valid_y0).float()
    w01 = (wx1 * wy0) * (valid_x1 & valid_y0).float()
    w10 = (wx0 * wy1) * (valid_x0 & valid_y1).float()
    w11 = (wx1 * wy1) * (valid_x1 & valid_y1).float()

    x0c, y0c = x0.clamp(0, W - 1), y0.clamp(0, H - 1)
    x1c, y1c = x1.clamp(0, W - 1), y1.clamp(0, H - 1)

    idx_00 = (y0c * W + x0c).reshape(N, num_out)
    idx_01 = (y0c * W + x1c).reshape(N, num_out)
    idx_10 = (y1c * W + x0c).reshape(N, num_out)
    idx_11 = (y1c * W + x1c).reshape(N, num_out)

    input_flat = input_tensor.reshape(N, C, H * W).permute(0, 2, 1).contiguous()

    return {
        "input_flat": input_flat,
        "idx_00": idx_00, "idx_01": idx_01, "idx_10": idx_10, "idx_11": idx_11,
        "w00": w00.reshape(N, num_out), "w01": w01.reshape(N, num_out),
        "w10": w10.reshape(N, num_out), "w11": w11.reshape(N, num_out),
        "N": N, "C": C, "H_out": H_out, "W_out": W_out, "num_out": num_out,
    }


def _pad_to_p_max(d):
    """Pad indices and weights to multiple of 128 (P_MAX)."""
    P_MAX = 128
    num_out = d["num_out"]
    pad_amt = (P_MAX - (num_out % P_MAX)) % P_MAX
    num_out_padded = num_out + pad_amt

    if pad_amt > 0:
        for key in ["idx_00", "idx_01", "idx_10", "idx_11"]:
            d[key] = torch.nn.functional.pad(d[key], (0, pad_amt), value=0)
        for key in ["w00", "w01", "w10", "w11"]:
            d[key] = torch.nn.functional.pad(d[key], (0, pad_amt), value=0.0)

    for key in ["idx_00", "idx_01", "idx_10", "idx_11"]:
        d[key] = d[key].unsqueeze(-1).to(torch.int32)
    for key in ["w00", "w01", "w10", "w11"]:
        d[key] = d[key].unsqueeze(-1)

    d["num_out_padded"] = num_out_padded
    d["pad_amt"] = pad_amt
    return d


def grid_sample_nki(input_tensor, grid):
    """
    Drop-in replacement for F.grid_sample(mode='bilinear', padding_mode='zeros',
    align_corners=False) using the NKI kernel.
    """
    d = _pad_to_p_max(_compute_bilinear_indices_weights(input_tensor, grid))
    N, C = d["N"], d["C"]
    H_out, W_out = d["H_out"], d["W_out"]
    num_out = d["num_out"]

    outputs = []
    for b in range(N):
        out_b = nki_bilinear_grid_sample_kernel(
            d["input_flat"][b],
            d["idx_00"][b], d["idx_01"][b], d["idx_10"][b], d["idx_11"][b],
            d["w00"][b], d["w01"][b], d["w10"][b], d["w11"][b],
        )
        out_b = out_b[:num_out, :].permute(1, 0).reshape(C, H_out, W_out)
        outputs.append(out_b)

    return torch.stack(outputs, dim=0)


print("NKI kernel and helpers defined.")

NKI monkey-patches applied.


NKI kernel and helpers defined.


## 3. Load Model, Save CPU Reference, and Patch

We save CPU reference outputs **before** patching so accuracy validation
doesn't require a second model download. Then we patch
`MultiScaleDeformableAttention.forward` to use our NKI kernel.

In [3]:
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
from transformers.models.rt_detr import modeling_rt_detr

model = RTDetrForObjectDetection.from_pretrained("PekingU/rtdetr_r101vd_coco_o365")
processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_r101vd_coco_o365")
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: PekingU/rtdetr_r101vd_coco_o365")
print(f"Parameters: {total_params / 1e6:.1f}M")


class RTDetrWrapper(nn.Module):
    """Thin wrapper that returns (logits, pred_boxes) tuple for tracing."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, pixel_values):
        outputs = self.model(pixel_values=pixel_values)
        return outputs.logits, outputs.pred_boxes

wrapper = RTDetrWrapper(model)
wrapper.eval()


# ── Save CPU reference BEFORE patching (uses original F.grid_sample) ──
print("Running CPU reference inference (20 random inputs)...")
torch.manual_seed(42)
cpu_ref_inputs = [torch.randn(1, 3, 640, 640) for _ in range(20)]
cpu_ref_logits, cpu_ref_boxes = [], []
with torch.no_grad():
    for inp in cpu_ref_inputs:
        logits, boxes = wrapper(inp)
        cpu_ref_logits.append(logits.clone())
        cpu_ref_boxes.append(boxes.clone())
print(f"  Saved {len(cpu_ref_logits)} CPU reference outputs.")


# ── Patch deformable attention to use NKI grid_sample ──

def patched_deformable_attention_forward_nki(
    self, value, value_spatial_shapes, value_spatial_shapes_list,
    level_start_index, sampling_locations, attention_weights, im2col_step=64,
):
    batch_size, _, num_heads, hidden_dim = value.shape
    _, num_queries, num_heads, num_levels, num_points, _ = sampling_locations.shape
    value_list = value.split(
        [h * w for h, w in value_spatial_shapes_list], dim=1
    )
    sampling_grids = 2 * sampling_locations - 1
    sampling_value_list = []
    for level_id, (height, width) in enumerate(value_spatial_shapes_list):
        value_l_ = (
            value_list[level_id]
            .flatten(2).transpose(1, 2)
            .reshape(batch_size * num_heads, hidden_dim, height, width)
        )
        sampling_grid_l_ = (
            sampling_grids[:, :, :, level_id].transpose(1, 2).flatten(0, 1)
        )
        sampling_value_l_ = grid_sample_nki(value_l_, sampling_grid_l_)
        sampling_value_list.append(sampling_value_l_)
    attention_weights = attention_weights.transpose(1, 2).reshape(
        batch_size * num_heads, 1, num_queries, num_levels * num_points
    )
    output = (
        (torch.stack(sampling_value_list, dim=-2).flatten(-2) * attention_weights)
        .sum(-1)
        .view(batch_size, num_heads * hidden_dim, num_queries)
    )
    return output.transpose(1, 2).contiguous()


modeling_rt_detr.MultiScaleDeformableAttention.forward = patched_deformable_attention_forward_nki
print("Deformable attention patched with NKI grid_sample.")

Model: PekingU/rtdetr_r101vd_coco_o365
Parameters: 76.6M
Running CPU reference inference (20 random inputs)...


  Saved 20 CPU reference outputs.
Deformable attention patched with NKI grid_sample.


## 4. Compile for Neuron

We compile two batch sizes:
- **BS=1**: Compiled in-process (first `trace()` call works fine)
- **BS=2**: Compiled via **subprocess isolation** (multiple `trace()` calls in the same
  process fail with NKI kernels)

Compilation takes ~5 minutes per variant. Uses FP32 (`--auto-cast none`) because
BF16 degrades box accuracy in RT-DETR's iterative decoder.
Compiled models are cached to `compiled/` and reloaded on subsequent runs.

In [4]:
example_bs1 = torch.randn(1, 3, 640, 640)

compiler_args = [
    "--verbose", "error",
    "--auto-cast", "none",
    "--optlevel", "3",
    "--model-type", "transformer",
    "--target", "inf2",
]

os.makedirs("compiled", exist_ok=True)

# Compile BS=1
bs1_path = "compiled/rtdetr_r101_nki_bs1.pt"
if os.path.exists(bs1_path):
    print(f"Loading pre-compiled BS=1 model from {bs1_path}")
    model_bs1 = torch.jit.load(bs1_path)
else:
    print("Compiling BS=1...")
    t0 = time.time()
    model_bs1 = torch_neuronx.trace(
        wrapper, example_bs1,
        compiler_workdir="compiled/workdir_bs1",
        compiler_args=compiler_args,
    )
    print(f"  BS=1 compiled in {time.time() - t0:.0f}s")
    torch.jit.save(model_bs1, bs1_path)
    print(f"  Saved to {bs1_path}")

Compiling BS=1...


The Python AST is located at: /tmp/klir_binaries/nki_bilinear_grid_sample_kernel_fs463a0x/nki_bilinear_grid_sample_kernelbxoiea5h_python_ast.klir
The KLR format is located at: final_klir_filepath='/tmp/klir_binaries/nki_bilinear_grid_sample_kernel_fs463a0x/nki_bilinear_grid_sample_kernelvbwn1f56.klir'
=========== warnings from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== messages from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== errors from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
The Python AST is located at: /tmp/klir_binaries/nki_bilinear_grid_sample_kernel_5yyfys4_/nki_bilinear_grid_sample_kernelcal60o7f_python_ast.klir
The KLR format is located at: final_klir_filepath='/tmp/klir_binaries/nki_bilinear_grid_sample_kernel_5yyfys4_/nki_bilinear_grid_sample_kernel3ro47_7s.klir'
=========== warnings from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== mess

The Python AST is located at: /tmp/klir_binaries/nki_bilinear_grid_sample_kernel_2ch_yhbc/nki_bilinear_grid_sample_kerneldn74sqm1_python_ast.klir
The KLR format is located at: final_klir_filepath='/tmp/klir_binaries/nki_bilinear_grid_sample_kernel_2ch_yhbc/nki_bilinear_grid_sample_kernele6qy41xr.klir'
=========== warnings from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== messages from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== errors from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
The Python AST is located at: /tmp/klir_binaries/nki_bilinear_grid_sample_kernel_0og_ta6s/nki_bilinear_grid_sample_kerneljjkdw9aw_python_ast.klir
The KLR format is located at: final_klir_filepath='/tmp/klir_binaries/nki_bilinear_grid_sample_kernel_0og_ta6s/nki_bilinear_grid_sample_kernel0p9xlsxr.klir'
=========== warnings from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== mess

The Python AST is located at: /tmp/klir_binaries/nki_bilinear_grid_sample_kernel_zlw6y05c/nki_bilinear_grid_sample_kernel06xvub_h_python_ast.klir
The KLR format is located at: final_klir_filepath='/tmp/klir_binaries/nki_bilinear_grid_sample_kernel_zlw6y05c/nki_bilinear_grid_sample_kernelpmkhay62.klir'
=========== warnings from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== messages from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== errors from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
The Python AST is located at: /tmp/klir_binaries/nki_bilinear_grid_sample_kernel_2tpzi9ib/nki_bilinear_grid_sample_kernel1apc0te7_python_ast.klir
The KLR format is located at: final_klir_filepath='/tmp/klir_binaries/nki_bilinear_grid_sample_kernel_2tpzi9ib/nki_bilinear_grid_sample_kernelh14xvbxw.klir'
=========== warnings from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== mess

The Python AST is located at: /tmp/klir_binaries/nki_bilinear_grid_sample_kernel_d3zu_0s7/nki_bilinear_grid_sample_kernel39hnfvbr_python_ast.klir
The KLR format is located at: final_klir_filepath='/tmp/klir_binaries/nki_bilinear_grid_sample_kernel_d3zu_0s7/nki_bilinear_grid_sample_kernel_2e_p4g2.klir'
=========== warnings from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== messages from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== errors from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
The Python AST is located at: /tmp/klir_binaries/nki_bilinear_grid_sample_kernel_w1i4knda/nki_bilinear_grid_sample_kerneldrj1ueen_python_ast.klir
The KLR format is located at: final_klir_filepath='/tmp/klir_binaries/nki_bilinear_grid_sample_kernel_w1i4knda/nki_bilinear_grid_sample_kernel2kesntpi.klir'
=========== warnings from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== mess

...

2026-04-14T14:30:31Z USER 5883 [Codegen]: debug_info_gen finished after 0.849 seconds


2026-04-14T14:30:32Z USER 5883 [ModuleForkPass]: codegen finished after 4.615 seconds
2026-04-14T14:30:32Z USER 5883 [BackendPassManager]: mod_parallel_pass finished after 4.718 seconds
2026-04-14T14:30:32Z USER 5883 [BackendPassManager]: Running hbm_usage
2026-04-14T14:30:32Z USER 5883 [BackendPassManager]: hbm_usage finished after 0.077 seconds
2026-04-14T14:30:32Z USER 5883 [BackendPassManager]: Running neff_packager


2026-04-14T14:30:42Z USER 5883 [BackendPassManager]: neff_packager finished after 9.801 seconds


2026-04-14T14:31:07Z USER 5673 [root]: Compiler status PASS


  BS=1 compiled in 342s
  Saved to compiled/rtdetr_r101_nki_bs1.pt


In [5]:
# Compile BS=2 via subprocess (multiple trace() calls in one process fail with NKI)
bs2_path = "compiled/rtdetr_r101_nki_bs2.pt"
if os.path.exists(bs2_path):
    print(f"Loading pre-compiled BS=2 model from {bs2_path}")
    model_bs2 = torch.jit.load(bs2_path)
else:
    print("Compiling BS=2 in subprocess (avoids multi-trace issue)...")
    # Write a self-contained compile script
    compile_script = '''
import os, sys, time
os.environ.setdefault("NEURON_PLATFORM_TARGET_OVERRIDE", "gen2")
import torch, torch.nn as nn, torch_neuronx
from transformers import RTDetrForObjectDetection
from transformers.models.rt_detr import modeling_rt_detr

# ── NKI setup ──
import nki, nki.language as nl, nki.isa as nisa
import nki.compile as _nki_compile

_orig_check_args = _nki_compile._check_args
def _patched_check_args(*args):
    def build_typename(a):
        t = type(a)
        module = getattr(t, "__module__", None)
        qualname = getattr(t, "__qualname__", None)
        if isinstance(a, torch.Tensor) and module and not module.startswith("torch."):
            return "torch.Tensor"
        return f"{module}.{qualname}"
    return {build_typename(arg) for arg in args}
_nki_compile._check_args = _patched_check_args

from nki.compiler.backends.neuron import TraceKernel as _TKM
_TKC = _TKM.TraceKernel
_orig_specialize = _TKC.specialize_and_call
def _patched_specialize(self, boundargs, output_path_prefix=None):
    def _to_regular(t):
        if isinstance(t, torch.Tensor) and type(t).__module__.startswith("torch_neuronx"):
            return torch.empty(t.shape, dtype=t.dtype)
        return t
    if boundargs.args:
        new_args = tuple(_to_regular(a) for a in boundargs.args)
        params = list(boundargs.signature.parameters.keys())
        for i, pn in enumerate(params):
            if i < len(new_args):
                boundargs.arguments[pn] = new_args[i]
    return _orig_specialize(self, boundargs, output_path_prefix)
_TKC.specialize_and_call = _patched_specialize

@nki.jit
def nki_bilinear_grid_sample_kernel(
    input_flat, idx_00, idx_01, idx_10, idx_11, w_00, w_01, w_10, w_11,
):
    num_out = idx_00.shape[0]
    C = input_flat.shape[1]
    P_MAX = 128
    num_tiles = num_out // P_MAX
    output = nl.ndarray((num_out, C), dtype=input_flat.dtype, buffer=nl.hbm)
    zero_bias = nl.ndarray((P_MAX, 1), dtype=nl.float32, buffer=nl.sbuf)
    nisa.memset(dst=zero_bias, value=0.0)
    for tile_idx in nl.affine_range(num_tiles):
        ts = tile_idx * P_MAX
        idx00_sb = nl.ndarray((P_MAX, 1), dtype=nl.uint32, buffer=nl.sbuf)
        idx01_sb = nl.ndarray((P_MAX, 1), dtype=nl.uint32, buffer=nl.sbuf)
        idx10_sb = nl.ndarray((P_MAX, 1), dtype=nl.uint32, buffer=nl.sbuf)
        idx11_sb = nl.ndarray((P_MAX, 1), dtype=nl.uint32, buffer=nl.sbuf)
        nisa.dma_copy(dst=idx00_sb, src=idx_00[ts:ts+P_MAX, 0:1])
        nisa.dma_copy(dst=idx01_sb, src=idx_01[ts:ts+P_MAX, 0:1])
        nisa.dma_copy(dst=idx10_sb, src=idx_10[ts:ts+P_MAX, 0:1])
        nisa.dma_copy(dst=idx11_sb, src=idx_11[ts:ts+P_MAX, 0:1])
        val00 = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)
        val01 = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)
        val10 = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)
        val11 = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)
        nisa.dma_copy(dst=val00, src=input_flat.ap(
            [[C, P_MAX], [1, C]], offset=0, vector_offset=idx00_sb, indirect_dim=0))
        nisa.dma_copy(dst=val01, src=input_flat.ap(
            [[C, P_MAX], [1, C]], offset=0, vector_offset=idx01_sb, indirect_dim=0))
        nisa.dma_copy(dst=val10, src=input_flat.ap(
            [[C, P_MAX], [1, C]], offset=0, vector_offset=idx10_sb, indirect_dim=0))
        nisa.dma_copy(dst=val11, src=input_flat.ap(
            [[C, P_MAX], [1, C]], offset=0, vector_offset=idx11_sb, indirect_dim=0))
        w00_sb = nl.ndarray((P_MAX, 1), dtype=nl.float32, buffer=nl.sbuf)
        w01_sb = nl.ndarray((P_MAX, 1), dtype=nl.float32, buffer=nl.sbuf)
        w10_sb = nl.ndarray((P_MAX, 1), dtype=nl.float32, buffer=nl.sbuf)
        w11_sb = nl.ndarray((P_MAX, 1), dtype=nl.float32, buffer=nl.sbuf)
        nisa.dma_copy(dst=w00_sb, src=w_00[ts:ts+P_MAX, 0:1])
        nisa.dma_copy(dst=w01_sb, src=w_01[ts:ts+P_MAX, 0:1])
        nisa.dma_copy(dst=w10_sb, src=w_10[ts:ts+P_MAX, 0:1])
        nisa.dma_copy(dst=w11_sb, src=w_11[ts:ts+P_MAX, 0:1])
        tmp = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)
        prod = nl.ndarray((P_MAX, C), dtype=nl.float32, buffer=nl.sbuf)
        nisa.activation(dst=tmp, op=nl.copy, data=val00, scale=w00_sb, bias=zero_bias)
        nisa.activation(dst=prod, op=nl.copy, data=val01, scale=w01_sb, bias=zero_bias)
        nisa.tensor_tensor(dst=tmp, data1=tmp, data2=prod, op=nl.add)
        nisa.activation(dst=prod, op=nl.copy, data=val10, scale=w10_sb, bias=zero_bias)
        nisa.tensor_tensor(dst=tmp, data1=tmp, data2=prod, op=nl.add)
        nisa.activation(dst=prod, op=nl.copy, data=val11, scale=w11_sb, bias=zero_bias)
        nisa.tensor_tensor(dst=tmp, data1=tmp, data2=prod, op=nl.add)
        nisa.dma_copy(dst=output[ts:ts+P_MAX, 0:C], src=tmp)
    return output

def _compute_bilinear_indices_weights(input_tensor, grid):
    N, C, H, W = input_tensor.shape
    H_out, W_out = grid.shape[1], grid.shape[2]
    num_out = H_out * W_out
    grid_x, grid_y = grid[..., 0], grid[..., 1]
    pix_x = ((grid_x + 1.0) * W - 1.0) / 2.0
    pix_y = ((grid_y + 1.0) * H - 1.0) / 2.0
    x0, y0 = pix_x.floor().long(), pix_y.floor().long()
    x1, y1 = x0 + 1, y0 + 1
    wx1, wy1 = pix_x - x0.float(), pix_y - y0.float()
    wx0, wy0 = 1.0 - wx1, 1.0 - wy1
    vx0 = (x0 >= 0) & (x0 < W); vy0 = (y0 >= 0) & (y0 < H)
    vx1 = (x1 >= 0) & (x1 < W); vy1 = (y1 >= 0) & (y1 < H)
    w00 = (wx0 * wy0) * (vx0 & vy0).float()
    w01 = (wx1 * wy0) * (vx1 & vy0).float()
    w10 = (wx0 * wy1) * (vx0 & vy1).float()
    w11 = (wx1 * wy1) * (vx1 & vy1).float()
    x0c, y0c = x0.clamp(0, W-1), y0.clamp(0, H-1)
    x1c, y1c = x1.clamp(0, W-1), y1.clamp(0, H-1)
    idx_00 = (y0c*W + x0c).reshape(N, num_out)
    idx_01 = (y0c*W + x1c).reshape(N, num_out)
    idx_10 = (y1c*W + x0c).reshape(N, num_out)
    idx_11 = (y1c*W + x1c).reshape(N, num_out)
    input_flat = input_tensor.reshape(N, C, H*W).permute(0, 2, 1).contiguous()
    return {"input_flat": input_flat,
            "idx_00": idx_00, "idx_01": idx_01, "idx_10": idx_10, "idx_11": idx_11,
            "w00": w00.reshape(N, num_out), "w01": w01.reshape(N, num_out),
            "w10": w10.reshape(N, num_out), "w11": w11.reshape(N, num_out),
            "N": N, "C": C, "H_out": H_out, "W_out": W_out, "num_out": num_out}

def _pad_to_p_max(d):
    P_MAX = 128
    num_out = d["num_out"]
    pad_amt = (P_MAX - (num_out % P_MAX)) % P_MAX
    num_out_padded = num_out + pad_amt
    if pad_amt > 0:
        for k in ["idx_00","idx_01","idx_10","idx_11"]:
            d[k] = torch.nn.functional.pad(d[k], (0, pad_amt), value=0)
        for k in ["w00","w01","w10","w11"]:
            d[k] = torch.nn.functional.pad(d[k], (0, pad_amt), value=0.0)
    for k in ["idx_00","idx_01","idx_10","idx_11"]:
        d[k] = d[k].unsqueeze(-1).to(torch.int32)
    for k in ["w00","w01","w10","w11"]:
        d[k] = d[k].unsqueeze(-1)
    d["num_out_padded"] = num_out_padded
    d["pad_amt"] = pad_amt
    return d

def grid_sample_nki(input_tensor, grid):
    d = _pad_to_p_max(_compute_bilinear_indices_weights(input_tensor, grid))
    N, C = d["N"], d["C"]
    H_out, W_out = d["H_out"], d["W_out"]
    num_out = d["num_out"]
    outputs = []
    for b in range(N):
        out_b = nki_bilinear_grid_sample_kernel(
            d["input_flat"][b],
            d["idx_00"][b], d["idx_01"][b], d["idx_10"][b], d["idx_11"][b],
            d["w00"][b], d["w01"][b], d["w10"][b], d["w11"][b])
        out_b = out_b[:num_out, :].permute(1, 0).reshape(C, H_out, W_out)
        outputs.append(out_b)
    return torch.stack(outputs, dim=0)

# ── Patch and compile ──
def patched_forward(self, value, value_spatial_shapes, value_spatial_shapes_list,
                    level_start_index, sampling_locations, attention_weights, im2col_step=64):
    batch_size, _, num_heads, hidden_dim = value.shape
    _, num_queries, _, num_levels, num_points, _ = sampling_locations.shape
    value_list = value.split([h*w for h,w in value_spatial_shapes_list], dim=1)
    sampling_grids = 2 * sampling_locations - 1
    sampling_value_list = []
    for lid, (height, width) in enumerate(value_spatial_shapes_list):
        vl = value_list[lid].flatten(2).transpose(1,2).reshape(batch_size*num_heads, hidden_dim, height, width)
        sg = sampling_grids[:,:,:,lid].transpose(1,2).flatten(0,1)
        sampling_value_list.append(grid_sample_nki(vl, sg))
    attention_weights = attention_weights.transpose(1,2).reshape(
        batch_size*num_heads, 1, num_queries, num_levels*num_points)
    output = (torch.stack(sampling_value_list, dim=-2).flatten(-2)*attention_weights
             ).sum(-1).view(batch_size, num_heads*hidden_dim, num_queries)
    return output.transpose(1,2).contiguous()

from transformers.models.rt_detr import modeling_rt_detr
modeling_rt_detr.MultiScaleDeformableAttention.forward = patched_forward

class RTDetrWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, pixel_values):
        o = self.model(pixel_values=pixel_values)
        return o.logits, o.pred_boxes

model = RTDetrForObjectDetection.from_pretrained("PekingU/rtdetr_r101vd_coco_o365")
model.eval()
w = RTDetrWrapper(model)
w.eval()

print("Compiling BS=2...")
t0 = time.time()
model_bs2 = torch_neuronx.trace(
    w, torch.randn(2, 3, 640, 640),
    compiler_workdir="compiled/workdir_bs2",
    compiler_args=["--verbose","error","--auto-cast","none","--optlevel","3","--model-type","transformer","--target","inf2"],
)
print(f"  BS=2 compiled in {time.time()-t0:.0f}s")
torch.jit.save(model_bs2, "compiled/rtdetr_r101_nki_bs2.pt")
print("  Saved to compiled/rtdetr_r101_nki_bs2.pt")
'''
    script_path = "compiled/_compile_bs2.py"
    with open(script_path, "w") as f:
        f.write(compile_script)
    
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, script_path],
        capture_output=True, text=True, timeout=900,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
        raise RuntimeError(f"BS=2 compilation failed (exit {result.returncode})")
    print(f"  Total BS=2 wall time: {time.time() - t0:.0f}s")
    model_bs2 = torch.jit.load(bs2_path)

Compiling BS=2 in subprocess (avoids multi-trace issue)...


Compiling BS=2...
The Python AST is located at: /tmp/klir_binaries/nki_bilinear_grid_sample_kernel_biymgkiq/nki_bilinear_grid_sample_kernel6xjd8fkg_python_ast.klir
The KLR format is located at: final_klir_filepath='/tmp/klir_binaries/nki_bilinear_grid_sample_kernel_biymgkiq/nki_bilinear_grid_sample_kernelx4e7xij9.klir'
=========== warnings from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== messages from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
=========== errors from kernel tracing __main__.nki_bilinear_grid_sample_kernel =========== 
The Python AST is located at: /tmp/klir_binaries/nki_bilinear_grid_sample_kernel_0kb521yy/nki_bilinear_grid_sample_kernelexo3ov3g_python_ast.klir
The KLR format is located at: final_klir_filepath='/tmp/klir_binaries/nki_bilinear_grid_sample_kernel_0kb521yy/nki_bilinear_grid_sample_kernelk10jda_l.klir'
=========== warnings from kernel tracing __main__.nki_bilinear_grid_sample_kernel ===========

## 5. Accuracy Validation

Compare Neuron model outputs against the CPU reference saved in Section 3
(original `F.grid_sample`, before patching). No second model download needed.
FP32 compilation should give near-perfect cosine similarity.

In [6]:
# Use the CPU reference saved in Section 3 (same seed, same inputs)
print("Accuracy Validation (20 random inputs, BS=1)")
print("=" * 65)

logits_cos_list, boxes_cos_list = [], []
for i, inp in enumerate(cpu_ref_inputs):
    with torch.no_grad():
        n_logits, n_boxes = model_bs1(inp)

    lcos = torch.nn.functional.cosine_similarity(
        cpu_ref_logits[i].flatten().unsqueeze(0).float(),
        n_logits.flatten().unsqueeze(0).float()
    ).item()
    bcos = torch.nn.functional.cosine_similarity(
        cpu_ref_boxes[i].flatten().unsqueeze(0).float(),
        n_boxes.flatten().unsqueeze(0).float()
    ).item()
    logits_cos_list.append(lcos)
    boxes_cos_list.append(bcos)

print(f"  Logits cosine sim: mean={np.mean(logits_cos_list):.6f}  min={np.min(logits_cos_list):.6f}")
print(f"  Boxes cosine sim:  mean={np.mean(boxes_cos_list):.6f}  min={np.min(boxes_cos_list):.6f}")
print("=" * 65)

Accuracy Validation (20 random inputs, BS=1)


  Logits cosine sim: mean=0.999989  min=0.999839
  Boxes cosine sim:  mean=0.999384  min=0.993953


## 6. Detection Demo

Run object detection on a real COCO image.

In [7]:
from PIL import Image
import requests

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
print(f"Image size: {image.size}")

inputs = processor(images=image, return_tensors="pt")
pixel_values = inputs["pixel_values"]

with torch.no_grad():
    logits, pred_boxes = model_bs1(pixel_values)

# Post-processing
scores = logits.sigmoid()
max_scores, labels = scores.max(dim=-1)

threshold = 0.3
keep = max_scores[0] > threshold
kept_scores = max_scores[0][keep]
kept_labels = labels[0][keep]
kept_boxes = pred_boxes[0][keep]

# Convert normalized boxes to pixel coordinates (center format -> corner format)
h, w = image.size[1], image.size[0]
bp = kept_boxes.clone()
bp[:, 0] *= w; bp[:, 1] *= h; bp[:, 2] *= w; bp[:, 3] *= h
x1 = bp[:, 0] - bp[:, 2] / 2
y1 = bp[:, 1] - bp[:, 3] / 2
x2 = bp[:, 0] + bp[:, 2] / 2
y2 = bp[:, 1] + bp[:, 3] / 2

coco_names = {
    0: "person", 1: "bicycle", 2: "car", 3: "motorcycle", 14: "bird",
    15: "cat", 16: "dog", 56: "chair", 57: "couch", 58: "potted plant",
    59: "bed", 60: "dining table", 62: "tv", 65: "remote",
}

print(f"\nDetected {len(kept_scores)} objects (threshold={threshold}):")
print("-" * 50)
for i in range(len(kept_scores)):
    label_id = kept_labels[i].item()
    name = coco_names.get(label_id, f"class_{label_id}")
    score = kept_scores[i].item()
    box = [x1[i].item(), y1[i].item(), x2[i].item(), y2[i].item()]
    print(f"  {name:15s}  score={score:.3f}  box=[{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]")

Image size: (640, 480)

Detected 5 objects (threshold=0.3):
--------------------------------------------------
  cat              score=0.959  box=[346, 25, 640, 373]
  remote           score=0.939  box=[40, 74, 175, 118]
  cat              score=0.967  box=[13, 55, 316, 473]
  remote           score=0.892  box=[334, 77, 370, 188]
  couch            score=0.944  box=[-0, -1, 640, 479]


## 7. Performance Benchmark

Three configurations, from simplest to best throughput:
- **Single core, BS=1**: Lowest latency per image
- **DP=2, BS=1/core**: 2 cores processing in parallel
- **DP=2, BS=2/core**: Best throughput (effective BS=4)

In [8]:
def benchmark(model, example_input, num_warmup=10, num_iters=100, label=""):
    for _ in range(num_warmup):
        model(example_input)

    times = []
    for _ in range(num_iters):
        t0 = time.perf_counter()
        model(example_input)
        elapsed = (time.perf_counter() - t0) * 1000
        times.append(elapsed)

    times = np.array(times)
    bs = example_input.shape[0]
    p50 = np.percentile(times, 50)
    p95 = np.percentile(times, 95)
    throughput = bs * 1000 / p50

    print(f"  {label:30s}  p50={p50:.1f}ms  p95={p95:.1f}ms  throughput={throughput:.1f} img/s")
    return {"p50": p50, "p95": p95, "throughput": throughput}


print("Performance Benchmark (100 iterations each)")
print("=" * 80)

# Single core BS=1
r_single = benchmark(model_bs1, torch.randn(1, 3, 640, 640), label="Single core BS=1")

# DP=2 BS=1/core
model_dp2_bs1 = torch_neuronx.DataParallel(model_bs1, device_ids=[0, 1], dim=0)
r_dp2_bs1 = benchmark(model_dp2_bs1, torch.randn(2, 3, 640, 640), label="DP=2 BS=1/core")
del model_dp2_bs1

# DP=2 BS=2/core (best config)
model_dp2_bs2 = torch_neuronx.DataParallel(model_bs2, device_ids=[0, 1], dim=0)
r_best = benchmark(model_dp2_bs2, torch.randn(4, 3, 640, 640), label="DP=2 BS=2/core (BEST)")
del model_dp2_bs2

print("=" * 80)

Performance Benchmark (100 iterations each)


  Single core BS=1                p50=48.4ms  p95=48.5ms  throughput=20.7 img/s


  DP=2 BS=1/core                  p50=54.6ms  p95=54.9ms  throughput=36.6 img/s


  DP=2 BS=2/core (BEST)           p50=109.4ms  p95=109.6ms  throughput=36.6 img/s


## 8. Results Summary

In [9]:
T4_FPS = 74.0

print("=" * 80)
print("RT-DETR R101 — Performance Summary (inf2.8xlarge, FP32)")
print("=" * 80)
print(f"{'Config':<35} {'p50 ms':>8} {'img/s':>8} {'vs T4':>8}")
print("-" * 80)
print(f"{'T4 GPU (reference)':<35} {'~14':>8} {T4_FPS:>8.1f} {'1.0x':>8}")
print(f"{'NKI BS=2 DP=2 (BEST)':<35} {r_best['p50']:>8.1f} {r_best['throughput']:>8.1f} {T4_FPS / r_best['throughput']:>7.1f}x")
print(f"{'NKI BS=1 DP=2':<35} {r_dp2_bs1['p50']:>8.1f} {r_dp2_bs1['throughput']:>8.1f} {T4_FPS / r_dp2_bs1['throughput']:>7.1f}x")
print(f"{'NKI BS=1 single core':<35} {r_single['p50']:>8.1f} {r_single['throughput']:>8.1f} {T4_FPS / r_single['throughput']:>7.1f}x")
print("=" * 80)
print()
print("Key Takeaways:")
print(f"  - Best throughput: {r_best['throughput']:.1f} img/s (DP=2, BS=2/core)")
print(f"  - T4 gap: {T4_FPS / r_best['throughput']:.1f}x (down from 8.6x with torch.gather)")
print(f"  - NKI custom kernel replaces torch.gather (DMA gather vs element-wise scatter)")
bs2_gain = r_best['throughput'] / r_dp2_bs1['throughput'] * 100 - 100
if bs2_gain > 1:
    print(f"  - BS=2 provides {bs2_gain:.0f}% throughput gain over BS=1")
else:
    print(f"  - BS=2 and BS=1 achieve similar throughput ({r_best['throughput']:.1f} vs {r_dp2_bs1['throughput']:.1f} img/s)")

RT-DETR R101 — Performance Summary (inf2.8xlarge, FP32)
Config                                p50 ms    img/s    vs T4
--------------------------------------------------------------------------------
T4 GPU (reference)                       ~14     74.0     1.0x
NKI BS=2 DP=2 (BEST)                   109.4     36.6     2.0x
NKI BS=1 DP=2                           54.6     36.6     2.0x
NKI BS=1 single core                    48.4     20.7     3.6x

Key Takeaways:
  - Best throughput: 36.6 img/s (DP=2, BS=2/core)
  - T4 gap: 2.0x (down from 8.6x with torch.gather)
  - NKI custom kernel replaces torch.gather (DMA gather vs element-wise scatter)
  - BS=2 and BS=1 achieve similar throughput (36.6 vs 36.6 img/s)
